# Customer Retention Intelligence — Monitoring & Observability (Phase 14)

## 1. Phase Objective

Add a lightweight, interview-appropriate monitoring layer on top of the existing
FastAPI inference service (Phases 7-13), covering:

- API request metrics (counts, latency, errors)
- Model inference metrics (predictions, outcomes)
- Model health
- Data-quality signals
- Data drift and prediction drift (Population Stability Index)
- Structured logging with request IDs
- Three new read-only `/monitoring/*` endpoints
- A Streamlit "Monitoring" tab

**No infrastructure was added**: no Prometheus/Grafana, no database, no external
monitoring platform. Storage is in-process memory only. **No model logic changed**:
the frozen Phase 7 model, Phase 8 SHAP, and Phase 9 recommendations are reused
completely unmodified -- this notebook only *observes* them.

This notebook uses FastAPI's `TestClient` (the same ASGI app object `api.main.app`,
including its full startup lifecycle) rather than a separately launched `uvicorn`
process, so it runs reproducibly end-to-end without port management. Docker-specific
verification is covered in `notebooks/13_dockerization.ipynb` and the Phase 14 report;
here the focus is the monitoring logic itself.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import JSON, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

from fastapi.testclient import TestClient

from src.models import baseline as bl
from api.main import app
from api.services import feature_row_to_payload

PROJECT_ROOT

/Users/dhananjaymandal/customer-retention-intelligence/.venv/lib/python3.14/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


PosixPath('/Users/dhananjaymandal/customer-retention-intelligence')

## 2. Existing Architecture (Unchanged)

```
Streamlit Dashboard -> HTTP -> FastAPI -> MLflow registered model (Phase 10)
    -> SHAP (Phase 8) / Next-Best-Action (Phase 9) -> JSON response
```

Phase 14 adds one middleware (request metrics + request ID + access logging) and
three new read-only endpoints. It does not touch `src/models/`, `src/recommendations/`,
or any existing endpoint's request/response contract.

In [2]:
client = TestClient(app)
client.__enter__()  # keeps the app's lifespan (and loaded model) alive for this notebook

health = client.get("/health")
print(f"GET /health -> {health.status_code}")
display(JSON(health.json()))

2026-09-23 22:34:11,833 INFO api.services: Model service ready: model=customer-churn-logistic-regression version=1 features=131 drift_monitor=True


2026-09-23 22:34:11,839 INFO api.access: request method=GET path=/health status=200 duration_ms=1.8 request_id=b6528554-98c7-4bab-a865-4148d057453f


2026-09-23 22:34:11,839 INFO httpx: HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"


GET /health -> 200


<IPython.core.display.JSON object>

## 3. API Monitoring: Request Metrics

`GET /monitoring/metrics` reports every HTTP request the API has served since it
started (including the `/health` call above).

In [3]:
metrics_before = client.get("/monitoring/metrics").json()
print(f"Total requests so far: {metrics_before['total_requests']}")
display(JSON(metrics_before))

2026-09-23 22:34:11,845 INFO api.access: request method=GET path=/monitoring/metrics status=200 duration_ms=1.0 request_id=9b96c328-157b-4691-869f-96e92e28ff27


2026-09-23 22:34:11,845 INFO httpx: HTTP Request: GET http://testserver/monitoring/metrics "HTTP/1.1 200 OK"


Total requests so far: 1


<IPython.core.display.JSON object>

## 4. Request Latency

The `latency_ms` block reports the mean latency since the process started, plus
p50/p95/p99 over the most recent 1000 requests (a bounded rolling window, so memory
use stays constant no matter how long the API has been running).

In [4]:
features = bl.load_feature_dataset()
sample_rows = features.sample(n=20, random_state=7)

for _, row in sample_rows.iterrows():
    payload = feature_row_to_payload(row)
    client.post("/predict", json=payload)

latency = client.get("/monitoring/metrics").json()["latency_ms"]
print("Latency after 20 /predict calls:")
display(JSON(latency))

2026-09-23 22:34:11,895 INFO api.main: POST /predict customer_id=C007457


2026-09-23 22:34:11,900 INFO api.access: request method=POST path=/predict status=200 duration_ms=5.8 request_id=48eb87f8-97aa-40c7-bd01-619355e48dd7


2026-09-23 22:34:11,900 INFO httpx: HTTP Request: POST http://testserver/predict "HTTP/1.1 200 OK"


2026-09-23 22:34:11,902 INFO api.main: POST /predict customer_id=C005728


2026-09-23 22:34:11,906 INFO api.access: request method=POST path=/predict status=200 duration_ms=4.4 request_id=0db25798-0d83-4f68-9ef3-4abefce58d87


2026-09-23 22:34:11,906 INFO httpx: HTTP Request: POST http://testserver/predict "HTTP/1.1 200 OK"


... (54 outputs left out)

2026-09-23 22:34:12,032 INFO api.access: request method=GET path=/monitoring/metrics status=200 duration_ms=0.5 request_id=9a9903c8-86e0-41f3-867a-fb2f5ef38c51


2026-09-23 22:34:12,032 INFO httpx: HTTP Request: GET http://testserver/monitoring/metrics "HTTP/1.1 200 OK"


Latency after 20 /predict calls:


<IPython.core.display.JSON object>

## 5. Error Monitoring

An invalid request (a required feature entirely omitted) is rejected with HTTP 422,
exactly as Phase 11 already does -- and is now also counted as a failed request.

In [5]:
failed_before = client.get("/monitoring/metrics").json()["failed_requests"]

bad_payload = feature_row_to_payload(sample_rows.iloc[0])
del bad_payload["age"]
response = client.post("/predict", json=bad_payload)
print(f"POST /predict (missing 'age') -> {response.status_code}")

failed_after = client.get("/monitoring/metrics").json()["failed_requests"]
print(f"failed_requests: {failed_before} -> {failed_after}")
assert failed_after == failed_before + 1

2026-09-23 22:34:12,038 INFO api.access: request method=GET path=/monitoring/metrics status=200 duration_ms=0.2 request_id=ff449f91-8811-4c5b-b163-2cd4fe0e5196


2026-09-23 22:34:12,038 INFO httpx: HTTP Request: GET http://testserver/monitoring/metrics "HTTP/1.1 200 OK"


2026-09-23 22:34:12,040 INFO api.access: request method=POST path=/predict status=422 duration_ms=0.5 request_id=a8a944ef-6a99-40ff-9902-f29810c4eb61


2026-09-23 22:34:12,040 INFO httpx: HTTP Request: POST http://testserver/predict "HTTP/1.1 422 Unprocessable Entity"


2026-09-23 22:34:12,041 INFO api.access: request method=GET path=/monitoring/metrics status=200 duration_ms=0.1 request_id=4be8be85-521b-4355-b50d-395948ebaf87


2026-09-23 22:34:12,041 INFO httpx: HTTP Request: GET http://testserver/monitoring/metrics "HTTP/1.1 200 OK"


POST /predict (missing 'age') -> 422
failed_requests: 0 -> 1


## 6. Prediction Monitoring

Separate from generic HTTP metrics: only the four inference endpoints update this.
`positive_predictions` / `negative_predictions` are split at the same 0.5
classification threshold Phase 5 established.

In [6]:
prediction_metrics = client.get("/monitoring/metrics").json()["predictions"]
display(JSON(prediction_metrics))

2026-09-23 22:34:12,045 INFO api.access: request method=GET path=/monitoring/metrics status=200 duration_ms=0.1 request_id=e9c618ad-8644-426b-8736-430af57304ae


2026-09-23 22:34:12,046 INFO httpx: HTTP Request: GET http://testserver/monitoring/metrics "HTTP/1.1 200 OK"


<IPython.core.display.JSON object>

## 7. Model Health

`GET /monitoring/health` reuses the same `ModelService` the four inference endpoints
already share -- it never loads a second copy of the model.

In [7]:
monitoring_health = client.get("/monitoring/health").json()
display(JSON(monitoring_health))

2026-09-23 22:34:12,049 INFO api.access: request method=GET path=/monitoring/health status=200 duration_ms=0.3 request_id=33844979-468e-4a0d-8f69-e7f06f91e7b3


2026-09-23 22:34:12,050 INFO httpx: HTTP Request: GET http://testserver/monitoring/health "HTTP/1.1 200 OK"


<IPython.core.display.JSON object>

## 8. Data Quality Monitoring

`average_null_field_rate` tracks how many of an *accepted* request's ~130 features
were null (legitimately missing, e.g. a customer with no support-ticket history --
the existing Phase 7 imputers handle this). This is additive: missing keys, wrong
types, and NaN/Infinity values are already rejected with HTTP 422 by the existing
Pydantic schema (Phase 11) before a request ever reaches this layer -- nothing here
re-validates that.

In [8]:
# A request with several features explicitly set to null -- still valid per the
# existing schema (Phase 11), and now visible in the null-rate metric.
sparse_payload = feature_row_to_payload(sample_rows.iloc[1])
nullable_features = ["avg_satisfaction_90d", "min_satisfaction_90d", "avg_resolution_hours_90d"]
for feature in nullable_features:
    sparse_payload[feature] = None

response = client.post("/predict", json=sparse_payload)
print(f"POST /predict with {len(nullable_features)} null features -> {response.status_code}")

null_rate = client.get("/monitoring/metrics").json()["predictions"]["average_null_field_rate"]
print(f"average_null_field_rate across all accepted predictions so far: {null_rate:.4f}")

2026-09-23 22:34:12,055 INFO api.main: POST /predict customer_id=C005728


2026-09-23 22:34:12,060 INFO api.access: request method=POST path=/predict status=200 duration_ms=5.4 request_id=9dc2732b-4ae5-41da-8f25-e6e7a5b20604


2026-09-23 22:34:12,060 INFO httpx: HTTP Request: POST http://testserver/predict "HTTP/1.1 200 OK"


2026-09-23 22:34:12,061 INFO api.access: request method=GET path=/monitoring/metrics status=200 duration_ms=0.1 request_id=1fd2c1ae-2cf8-4f03-9d86-95ed72c15c2f


2026-09-23 22:34:12,061 INFO httpx: HTTP Request: GET http://testserver/monitoring/metrics "HTTP/1.1 200 OK"


POST /predict with 3 null features -> 200
average_null_field_rate across all accepted predictions so far: 0.0687


## 9. Data Drift

Population Stability Index (PSI) for a small set of numeric features -- chosen as the
**top-8 by Phase 8's own SHAP importance ranking** (`reports/shap_feature_importance.csv`),
not an arbitrary list -- plus the 4 categorical features, each compared against the
**Phase 7 TRAIN split** (the data the registered model was actually fit on).

In [9]:
for _, row in features.sample(n=60, random_state=21).iterrows():
    client.post("/predict", json=feature_row_to_payload(row))

drift = client.get("/monitoring/drift").json()
print(f"Baseline: {drift['baseline_source']}")
print(f"Monitored numeric features: {drift['monitored_numeric_features']}")
print(f"Monitored categorical features: {drift['monitored_categorical_features']}")

feature_drift_df = pd.DataFrame(drift["feature_drift"])
feature_drift_df

2026-09-23 22:34:12,068 INFO api.main: POST /predict customer_id=C005690


2026-09-23 22:34:12,073 INFO api.access: request method=POST path=/predict status=200 duration_ms=4.7 request_id=c7fa30b7-8f27-4b6b-baaa-8e20601d401f


2026-09-23 22:34:12,073 INFO httpx: HTTP Request: POST http://testserver/predict "HTTP/1.1 200 OK"


2026-09-23 22:34:12,075 INFO api.main: POST /predict customer_id=C004133


2026-09-23 22:34:12,079 INFO api.access: request method=POST path=/predict status=200 duration_ms=4.4 request_id=30eea757-936a-4f15-a26b-8eb501f24738


2026-09-23 22:34:12,079 INFO httpx: HTTP Request: POST http://testserver/predict "HTTP/1.1 200 OK"


... (174 outputs left out)

2026-09-23 22:34:12,416 INFO api.access: request method=GET path=/monitoring/drift status=200 duration_ms=2.0 request_id=e774489c-2dd9-496a-a79f-4ec7a4b23e4a


2026-09-23 22:34:12,417 INFO httpx: HTTP Request: GET http://testserver/monitoring/drift "HTTP/1.1 200 OK"


Baseline: Phase 7 TRAIN split of data/processed/customer_features.parquet
Monitored numeric features: ['avg_pages_viewed_90d', 'active_days_90d', 'avg_active_minutes_90d', 'avg_feature_usage_count_previous30d', 'avg_active_minutes_previous30d', 'txn_amount_sum_180d', 'avg_session_minutes_previous30d', 'avg_pages_viewed_recent30d']
Monitored categorical features: ['country', 'plan', 'contract_type', 'acquisition_channel']


,feature,psi,status,sample_size
0,avg_pages_viewed_90d,0.1471,moderate_shift,81
1,active_days_90d,0.1799,moderate_shift,81
2,avg_active_minutes_90d,0.2860,significant_shift,81
3,avg_feature_usage_count_previous30d,0.2059,moderate_shift,77
4,avg_active_minutes_previous30d,0.0797,no_significant_shift,77
5,txn_amount_sum_180d,0.1431,moderate_shift,81
6,avg_session_minutes_previous30d,0.1038,moderate_shift,77
7,avg_pages_viewed_recent30d,0.0239,no_significant_shift,80
8,country,0.0044,no_significant_shift,81
9,plan,0.0083,no_significant_shift,81


## 10. Prediction Drift

Same PSI metric, applied to the model's own predicted-probability distribution,
compared against predictions on the **VALIDATION** split (out-of-sample, scored once
at API startup) -- not TRAIN, so the baseline reflects genuinely unseen data.

In [10]:
prediction_drift = drift["prediction_drift"]
display(JSON(prediction_drift))

print(f"\nPSI thresholds: {drift['psi_thresholds']}")
print(f"Drifted feature count: {drift['drifted_feature_count']}")
print(f"Drifted features: {drift['drifted_features']}")

<IPython.core.display.JSON object>


PSI thresholds: {'moderate': 0.1, 'significant': 0.25}
Drifted feature count: 6
Drifted features: ['avg_pages_viewed_90d', 'active_days_90d', 'avg_active_minutes_90d', 'avg_feature_usage_count_previous30d', 'txn_amount_sum_180d', 'avg_session_minutes_previous30d']


## 11. Example Monitoring Outputs (Full Responses)

In [11]:
print("=== GET /monitoring/metrics ===")
display(JSON(client.get("/monitoring/metrics").json()))

2026-09-23 22:34:12,428 INFO api.access: request method=GET path=/monitoring/metrics status=200 duration_ms=0.1 request_id=4f509703-c42e-49df-a28f-2c5c67a15b47


2026-09-23 22:34:12,429 INFO httpx: HTTP Request: GET http://testserver/monitoring/metrics "HTTP/1.1 200 OK"


=== GET /monitoring/metrics ===


<IPython.core.display.JSON object>

In [12]:
print("=== GET /monitoring/health ===")
display(JSON(client.get("/monitoring/health").json()))

2026-09-23 22:34:12,432 INFO api.access: request method=GET path=/monitoring/health status=200 duration_ms=0.1 request_id=fedd04ba-7f13-4ff1-a688-346397d670c8


2026-09-23 22:34:12,432 INFO httpx: HTTP Request: GET http://testserver/monitoring/health "HTTP/1.1 200 OK"


=== GET /monitoring/health ===


<IPython.core.display.JSON object>

In [13]:
print("=== GET /monitoring/drift ===")
display(JSON(client.get("/monitoring/drift").json()))

2026-09-23 22:34:12,438 INFO api.access: request method=GET path=/monitoring/drift status=200 duration_ms=1.9 request_id=d9988395-4229-4d27-83d4-036bb2db035f


2026-09-23 22:34:12,438 INFO httpx: HTTP Request: GET http://testserver/monitoring/drift "HTTP/1.1 200 OK"


=== GET /monitoring/drift ===


<IPython.core.display.JSON object>

## 12. Interpretation

- **Request/latency metrics** confirm the API is serving requests quickly (single-digit
  to low-double-digit milliseconds per prediction in this environment) and that a
  malformed request is both rejected (422) *and* visible in `failed_requests` --
  the two concerns (correctness and observability) reinforce each other.
- **Prediction metrics** show a roughly even split between positive/negative
  predictions at the 0.5 threshold, consistent with Phase 7's `class_weight="balanced"`
  training choice shifting predicted probabilities upward relative to the true ~1.2%
  churn rate (see Phase 12's dashboard note) -- this is expected model behavior, not
  a monitoring anomaly.
- **Drift results** in this run reflect real, if modest, sample-to-sample variation:
  some monitored features may show `moderate_shift` or `significant_shift` purely from
  drawing a different random subset of the dataset than the training/validation split
  did. This is a genuine, useful illustration of PSI's sample-size sensitivity (see
  Limitations) -- it is not evidence that anything is actually wrong with the model.
  **A drift signal here means "look into it," never "the model is broken."**

## 13. Limitations

- **In-memory only**: every counter and rolling window shown above resets on API
  restart. There is no persisted metrics history.
- **PSI is sample-size sensitive**: with 10 histogram bins, comparison windows under
  roughly 100 observations can show a noisy PSI even without real drift (demonstrated
  directly in `tests/test_monitoring.py`'s no-drift test, which needed 400 samples to
  be statistically stable rather than the 30-sample minimum the API itself requires
  before reporting anything at all).
- **Fixed baselines**: the TRAIN/VALIDATION baselines are computed once at API startup
  from whatever `data/processed/customer_features.parquet` and the registered model
  currently are -- they do not update automatically.
- **No performance-degradation detection**: drift is a distributional signal, not a
  measurement of actual prediction quality, which would require ground-truth
  `churn_next_30_days` outcomes not available at inference time.

## 14. Production Considerations (Not Implemented Here)

For a real production deployment, this lightweight layer would typically be extended
with (described for completeness, deliberately **not built** in this phase, per the
"don't overengineer an interview-scale project" scope):

- A persistent metrics/time-series store (e.g. Prometheus + Grafana, or a managed
  APM) so history survives restarts and supports trend/alerting dashboards.
- Automated alerting on drift thresholds and error-rate spikes, routed to an on-call
  system.
- A scheduled job that periodically re-scores a fresh labeled sample (once
  ground-truth churn outcomes become available) to directly measure realized
  performance, not just distributional drift.
- Baseline refresh policy: deciding when/how the TRAIN/VALIDATION baselines themselves
  should be recomputed as the business and customer population evolve.
- Multi-instance aggregation if the API is ever horizontally scaled (this
  implementation's in-memory metrics are per-process, not shared across replicas).

In [14]:
client.__exit__(None, None, None)
print("TestClient stopped.")

TestClient stopped.
